# Minio Tutorial

This notebook provides an introductory tutorial to the [Minio Docker Image](https://hub.docker.com/r/minio/minio).
- You can also see the [Minio Documentation](https://min.io/docs/minio/linux/index.html).
- You can also visit the [Official Minio Website](https://min.io).

---

## Connect to a Jupyter Python Kernel

- Let's connect to our Jupyter Python Kernel we registered in a previous notebook.
  - Make sure you have chosen the `.NET Interactive` kernel in the top right of this notebook.
  - Make sure the cell below is using `csharp - C# Script Code`.
  - Execute the cell below.
    - You should see a message similar to: `Kernel added: #!python`
    - or: `Error: (1,1): error DNI211: The kernel name or alias 'python' is already in use.`

In [2]:
#!connect jupyter --kernel-name python --kernel-spec .conda

The `#!connect jupyter` feature is in preview. Please report any feedback or issues at https://github.com/dotnet/interactive/issues/new/choose.

Kernel added: #!python

---

## Start a Minio Container

The [Minio Docker Image](https://hub.docker.com/r/minio/minio) is fairly easy to use when creating a container.
- First we create a Docker bridge network called `minio` with the command `docker create network minio`.
- We use the command `docker run -d --rm --name minio --network minio -p 9000:9000 -p 9001:9001 -e MINIO_ROOT_USER=minio_user -e MINIO_ROOT_PASSWORD=minio_password -e MINIO_ADDRESS=9000 -e MINIO_CONSOLE_ADDRESS=9001 -v minio_data:/data minio/minio:RELEASE.2024-10-13T13-34-11Z minio server /data --address "0.0.0.0:9000" --console-address ":9001"` to:
  - Create and start a container (`docker run`).
  - Instantiate the container based on the Rabbit MQ Image (`minio/minio:RELEASE.2024-10-13T13-34-11Z`).
  - Run the container in detached mode (`-d`).
  - Automatically delete the container when it is stopped (`--rm`).
- We also:
  - Set `--name` to e.g. `minio` (this name can be used as the DNS name in network communication).
  - Set `--network` to `minio` to connect the container to the `minio` Docker Bridge network.
  - Map host port `9000` to container port `9000` using the `-p` flag.
    - This port is used to communicate with Minio.
  - Map host port `9001` to container port `9001` using the `-p` flag.
    - This port is Minio's management port, which we use to access Minio's Web UI.
  - Set four environment variables using the `-e` flag:
    - `MINIO_ROOT_USER=minio_user` sets the name of the `root user`.
    - `MINIO_ROOT_PASSWORD=minio_password` sets the `password` for the `root user`.
    - `MINIO_ADDRESS=9000` sets the communication port Minio should listen on.
    - `MINIO_CONSOLE_ADDRESS=9001`  sets the management port Minio should listen on.
  - Map named volume `minio_data` (on the host) to container folder `/data` so that our Blobs (files) are preserved when we delete the container.
  - Call the command `minio server /data --address "0.0.0.0:9000" --console-address ":9001"` when the Minio container starts.
    - This starts the Minio server inside the container.

Minio can be reached on:
- `minio:9000` if accessed via the `minio` Docker Bridge network.
- `localhost:9000` if accessed from the host.
- `localhost:9001` to access Minio's Management Web UI.

In [3]:
!docker network create minio
!docker run -d --rm --name minio --network minio -p 9000:9000 -p 9001:9001 -e MINIO_ROOT_USER=minio_user -e MINIO_ROOT_PASSWORD=minio_password -e MINIO_ADDRESS=9000 -e MINIO_CONSOLE_ADDRESS=9001 -v minio_data:/data minio/minio:RELEASE.2024-10-13T13-34-11Z minio server /data --address "0.0.0.0:9000" --console-address ":9001"

!docker network ls
!docker ps

83343ce0a41651448a7eae84fde5404fa97b65161123f902212d6d9e117571e6
f55beac3f63c60cd3b9df3d723815d67ada4aaf40ef27dc179f8d74368f9bf2f
NETWORK ID     NAME             DRIVER    SCOPE
83343ce0a416   minio            bridge    local
CONTAINER ID   IMAGE                                      COMMAND                  CREATED        STATUS        PORTS                              NAMES
f55beac3f63c   minio/minio:RELEASE.2024-10-13T13-34-11Z   "/usr/bin/docker-entâ€¦"   1 second ago   Up 1 second   0.0.0.0:9000-9001->9000-9001/tcp   minio


---

## Start a Minio-MC Container

The [Minio MC Docker Image](https://hub.docker.com/r/minio/mc) provides Minio CLI tools for working with Minio.
- We'll use this container to configure Minio, create a sample bucket andseed it with a sample video file.
- We use the command `docker run -d --rm --name minio-mc --network minio -v .\fixtures\minio_initialize_and_seed.sh:/docker-entrypoint-initdb.d/minio_import.sh -v .\fixtures\SampleVideo_1280x720_1mb.mp4:/tmp/videos/SampleVideo_1280x720_1mb.mp4 --entrypoint bash minio/mc:RELEASE.2024-10-08T09-37-26Z -c "/docker-entrypoint-initdb.d/minio_import.sh"` to:
  - Create and start a container (`docker run`).
  - Instantiate the container based on the Minio MC Docker Image (`minio/mc:RELEASE.2024-10-08T09-37-26Z`).
  - Run the container in detached mode (`-d`).
  - Automatically delete the container when it is stopped (`--rm`).
- We also:
  - Set `--name` to e.g. `minio-mc`.
  - Bind mount the file `.\fixtures\minio_initialize_and_seed.sh` (on the host) to container file `/docker-entrypoint-initdb.d/minio_import.sh`, which contains a bash script for setting up Minio, creating a sample bucket called "videos", and uploading a sample video file to the bucket.
  - Bind mount the file `.\fixtures\SampleVideo_1280x720_1mb.mp4` (on the host) to container file `/tmp/videos/SampleVideo_1280x720_1mb.mp4`, which is the sample video file that will be uploaded to the sample bucket.
  - Set `--entrypoint` to  `bash`, i.e. when the container starts, it will run the command `bash` inside the container.
  - Pass the arguments `-c "/docker-entrypoint-initdb.d/minio_import.sh"` to the `bash` command when the Minio container starts.
    - This will run the bash script (that we mapped from the host to the container above) inside the container.

Let's look at the bash script below:
- It waits until it can connect to the Minio container (`curl -I http://minio:9000/minio/health/live`).
- When it can connect to Minio, it loggs in `/usr/bin/mc alias set minio http://minio:9000 minio_user minio_password`.
- Once logged in, it creates a bucket called `videos` (`/usr/bin/mc mb minio/videos`).
- It sets the bucket's access to `public` so anyone can access it (`/usr/bin/mc anonymous set public minio/videos`).
- Then it uploads a sample video file to the bucket (`/usr/bin/mc put /tmp/videos/SampleVideo_1280x720_1mb.mp4 minio/videos`).
- Next, it creates an admin user (`/usr/bin/mc admin user add minio Y42xk0CJlYV2fnShCtBP kCA01XftotivPbU9TRJ5Nrr5VMjFZNEhQhtvfWyY`).
  - `Y42xk0CJlYV2fnShCtBP` is the username.
  - `kCA01XftotivPbU9TRJ5Nrr5VMjFZNEhQhtvfWyY` is the password.
- It also grants the user readwrite permissions (`/usr/bin/mc admin policy attach minio readwrite --user Y42xk0CJlYV2fnShCtBP`).
- Finally, it exits the bash shell, which will cause the Minio-MC container to stop.

  ```csharp
  #!/bin/bash

  # Wait for Minio to become available
  while [ ! $(mc alias set minio http://minio:9000 minio_user minio_password) ]
  # while [ ! $(curl -I http://minio:9000/minio/health/live) ]
  do
    echo 'Waiting for minio to start up ...'
    sleep 0.1
  done
  sleep 5

  # Create public bucket with object
  /usr/bin/mc alias set minio http://minio:9000 minio_user minio_password
  /usr/bin/mc mb minio/videos
  /usr/bin/mc anonymous set public minio/videos
  /usr/bin/mc put /tmp/videos/SampleVideo_1280x720_1mb.mp4 minio/videos

  # Create user with readwrite and deleteobject access
  /usr/bin/mc admin user add minio Y42xk0CJlYV2fnShCtBP kCA01XftotivPbU9TRJ5Nrr5VMjFZNEhQhtvfWyY
  /usr/bin/mc admin policy attach minio readwrite --user Y42xk0CJlYV2fnShCtBP

  # Exit shell (so that container terminates)
  exit 0
  ```

To access the Minio bucket called `videos` we need to use:

- `minio:9000` which is the URL and Port used to access Minio.
- `Y42xk0CJlYV2fnShCtBP` which is user's username.
- `kCA01XftotivPbU9TRJ5Nrr5VMjFZNEhQhtvfWyY` which is the user's password.
- `videos` which is the name of the bucket.

In [4]:
!docker run -d --rm --name minio-mc --network minio -v .\fixtures\minio_initialize_and_seed.sh:/docker-entrypoint-initdb.d/minio_import.sh -v .\fixtures\SampleVideo_1280x720_1mb.mp4:/tmp/videos/SampleVideo_1280x720_1mb.mp4 --entrypoint bash minio/mc:RELEASE.2024-10-08T09-37-26Z -c "/docker-entrypoint-initdb.d/minio_import.sh"
# !docker run -d --rm --name minio-mc --network minio -v .\fixtures\minio_initialize.sh:/docker-entrypoint-initdb.d/minio_import.sh -v .\fixtures\SampleVideo_1280x720_1mb.mp4:/tmp/videos/SampleVideo_1280x720_1mb.mp4 --entrypoint bash minio/mc:RELEASE.2024-10-08T09-37-26Z -c "/docker-entrypoint-initdb.d/minio_import.sh"

!docker ps

0eaf42f9e700dc2fbeb06925294679179e8016ba94415976c2cfa45b84f4ed9d
CONTAINER ID   IMAGE                                      COMMAND                  CREATED                  STATUS                  PORTS                              NAMES
0eaf42f9e700   minio/mc:RELEASE.2024-10-08T09-37-26Z      "bash -c /docker-entâ€¦"   Less than a second ago   Up Less than a second                                      minio-mc
f55beac3f63c   minio/minio:RELEASE.2024-10-13T13-34-11Z   "/usr/bin/docker-entâ€¦"   16 seconds ago           Up 15 seconds           0.0.0.0:9000-9001->9000-9001/tcp   minio


---

## Install VSCode Extensions for working with Minio

- There aren't really any good VSCode Extensions for Working with Minio, but there's one extension below.
  - [MinIO](https://marketplace.visualstudio.com/items?itemName=seriousbenentertainment.minio) provides very basic functionality when working with Minio.

Let's use the [MinIO](https://marketplace.visualstudio.com/items?itemName=seriousbenentertainment.minio) extension.

In [5]:
!code --install-extension seriousbenentertainment.minio --force

Installing extensions...
Extension 'seriousbenentertainment.minio' is already installed.


---

## Use the Minio Extension

Let's connect to the Minio `videos` bucket we set up above using the Minio Extension:
- Open VSCode's settings `File -> Preferences -> Settings` (or Windows/Linux: `Ctrl + ,`, Mac: `Cmd + ,`).
- Enter `minio` into the search bar, and configure the following settings:
  - Set `Minio > Minio > Server: Address` to `http://127.0.0.1:9000`
  - Set `Minio > Minio > Credential: Access Key` to `minio_user`
  - Set `Minio > Minio > Credential: Secret Key` to `minio_password`
  - Set `Minio > Minio > Upload: Bucket Name` to `videos`
  - Set `Minio > Minio > Download: Directory` to your Downloads folder e.g.:
    - Windows: `%USERPROFILE%\Downloads`
    - Linux/Mac: `$HOME/Downloads`
- Now open the `Minio` Extension.
  - Click the `Refresh` icon (circlular arrow). 
  - You should now see the sample video `SampleVideo_1280x720_1mb.mp4` in the `videos` bucket.

<img src="../notebook_images/minio_extension.png" width="250" />

**Unfortunately you can't do much more using the Minio VSCode Extension, so let's use Minio's Web UI.**

---

## Use the Minio Web UI

Let's connect to Minio's Web UI:
- Visit http://localhost:9001
- Enter `minio_user` as the username.
- Enter `minio_password` as the password.
- Click the `Login` button.

<img src="../notebook_images/minio_webui_login.png" width="600" />

- The `Object Browser` gives an overview of Minio buckets and objects.
  - We see there is a bucket called `videos` with `1` object in it.
  - Click the `videos` bucket.

<img src="../notebook_images/minio_webui_object_browser.png" width="600" />

- The `videos` bucket view allows us to upload files to it as objects, and to delete objects.
  - We see that the `videos` bucket currently contains one object called `SampleVideo_1280x720_1mb.mp4`.
  - Click the `Buckets` tab in the left margin.

<img src="../notebook_images/minio_webui_object_browser_videos.png" width="600" />

- The `Buckets` tab allows us to view, create and delete buckets.
  - Click the `videos` bucket.

<img src="../notebook_images/minio_webui_buckets.png" width="600" />

- Detailed information about the `videos` bucket is shown, including a button for deleting the bucket.
  - Notice the `Access Policy` is set to `Public`.
  - Click the `Access` option.

<img src="../notebook_images/minio_webui_buckets_videos.png" width="600" />

- Under `Access` choose the `Users` tab.
  - Notice the user with username `Y42xk0CJlYV2fnShCtBP` has been granted access to this bucket.
  - Click on the username `Y42xk0CJlYV2fnShCtBP`.

<img src="../notebook_images/minio_webui_buckets_videos_access_users.png" width="600" />

- For user `Y42xk0CJlYV2fnShCtBP`, select the `Policies` option.
  - Notice the `Y42xk0CJlYV2fnShCtBP` has been granted `readwrite` access.

<img src="../notebook_images/minio_webui_buckets_videos_access_users_policies.png" width="600" />

---

## Install NuGet Packages

- Let's install the required NuGet packages to use the RabbitMQ Client.
  - In a `.NET Interactive` Ployglot Notebook, we use the `#r` magic command in a `csharp - C# Script Code` cell to install NuGet packages.

In [6]:
// Install the required NuGet packages for Minio
#r "nuget: Minio, 6.0.3"

Installed Packages Minio, 6.0.3

---

## Connect to Minio

To connect to Minio, we create an instance of the `MinioClient` with the Minio endpoint, username and password.

```csharp
string STORAGE_ENDPOINT = "localhost:9000";
string STORAGE_ACCESS_KEY = "Y42xk0CJlYV2fnShCtBP"; // username
string STORAGE_SECRET_KEY = "kCA01XftotivPbU9TRJ5Nrr5VMjFZNEhQhtvfWyY"; // password

IMinioClient minioClient = new MinioClient()
    .WithEndpoint(STORAGE_ENDPOINT)
    .WithCredentials(STORAGE_ACCESS_KEY, STORAGE_SECRET_KEY)
    .WithSSL(false)
    .Build();
```

In [7]:
using Minio;
using Minio.DataModel;
using Minio.DataModel.Args;
using Minio.Exceptions;
using System.IO;

string STORAGE_ENDPOINT = "localhost:9000";
string STORAGE_ACCESS_KEY = "Y42xk0CJlYV2fnShCtBP";
string STORAGE_SECRET_KEY = "kCA01XftotivPbU9TRJ5Nrr5VMjFZNEhQhtvfWyY";

// Connect to Minio
IMinioClient minioClient = new MinioClient()
    .WithEndpoint(STORAGE_ENDPOINT)
    .WithCredentials(STORAGE_ACCESS_KEY, STORAGE_SECRET_KEY)
    .WithSSL(false)
    .Build();

---

## Create a Bucket

To create a bucket (equivalent to a folder in a file system), we:
- First make sure the bucket doesn't already exist.

    ```csharp
    string STORAGE_BUCKET_NAME = "videos2";
    BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
    bool bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);
    ```

- Then we create a `MakeBucketArgs` object.

    ```csharp
    MakeBucketArgs makeBucketArgs = new MakeBucketArgs().WithBucket(STORAGE_BUCKET_NAME);
    ```

- Finally, we call the Minio client's `MakeBucketAsync()` method, passing in the `MakeBucketArgs` object.

    ```csharp
    await minioClient.MakeBucketAsync(makeBucketArgs);
    ```

In [8]:
// Minio bucket name
string STORAGE_BUCKET_NAME = "videos2";

// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
bool bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Create the Minio bucket if it doesn't exist
if(!bucketExists)
{
    // Create the Minio bucket.
    MakeBucketArgs makeBucketArgs = new MakeBucketArgs().WithBucket(STORAGE_BUCKET_NAME);
    await minioClient.MakeBucketAsync(makeBucketArgs);
}

---

## Configure a Policy for a Bucket

Policies determine what a user can do, including operations on buckets, and also if public access to the bucket is allowed.

A policy is defined as a JSON document:
- The `Version` of the JSON document is given, followed by a `Statement` array.
- Each `Statement` in the array contains the following properties:
  - `"Effect"` states if permissions are being granted or denied.
    - `"Allow"` means permissions are being granted.
  - `"Principal"` states the users the permissions will be applied to.
    - `{"AWS":["*"]}` means ALL users.
  - `"Action"` states the actions (permissions).
    - `s3:GetBucketLocation` is for getting a bucket's location.
    - `s3:AbortMultipartUpload` is for aborting a multi-part upload.
    - `s3:ListBucketMultipartUploads` is for listing multi-part uploads for buckets.
    - `s3:ListMultipartUploadParts` is for listing multi-part uploads.
    - `s3:ListBucket` is for listing buckets.
    - `s3:PutObject` is for uploading an object to a bucket.
    - `s3:GetObject` is for downloading an object from a bucket.
    - `s3:DeleteObject` is for deleting an object in a bucket.
  - `"Resource"` states the resources the permissions will be applied to:
    - `"arn:aws:s3:::STORAGE_BUCKET_NAME"` is a bucket called `STORAGE_BUCKET_NAME`.
    - `"arn:aws:s3:::STORAGE_BUCKET_NAME/*"` is the `STORAGE_BUCKET_NAME` bucket and all its sub-buckets, objects, etc.

```json
{
    "Version":"2012-10-17",
    "Statement":[
        {
            "Effect":"Allow",
            "Principal":{"AWS":["*"]},
            "Action":[
                "s3:GetBucketLocation",
                "s3:ListBucket",
                "s3:ListBucketMultipartUploads"
            ],
            "Resource":["arn:aws:s3:::STORAGE_BUCKET_NAME"]
        },
        {
            "Effect":"Allow",
            "Principal":{"AWS":["*"]},
            "Action":[
                "s3:AbortMultipartUpload",
                "s3:DeleteObject",
                "s3:GetObject",
                "s3:ListMultipartUploadParts",
                "s3:PutObject"
            ],
            "Resource":["arn:aws:s3:::STORAGE_BUCKET_NAME/*"]
        }
    ]
}
```

To set a policy we:
- First create a policy JSON document as a string (e.g. store the JSON document example above in a string).

    ```csharp
    string publicPolicy = ...
    ```

- Then we create a `SetPolicyArgs` object with our policy and bucket name as inputs.

    ```csharp
    SetPolicyArgs setPolicyArgs = new SetPolicyArgs().WithPolicy(publicPolicy).WithBucket(STORAGE_BUCKET_NAME);
    ```

- Finally, we call the Minio client's `SetPolicyAsync()` method with the `SetPolicyArgs` object as an argument.

    ```csharp
    await minioClient.SetPolicyAsync(setPolicyArgs);
    ```

- We can also get the policies that have been applied to a bucket and print them out.

    ```csharp
    GetPolicyArgs getPolicyArgs = new GetPolicyArgs().WithBucket(STORAGE_BUCKET_NAME);
    var policy = await minioClient.GetPolicyAsync(getPolicyArgs);
    Console.WriteLine($"policy: {policy}");
    ```

In [9]:
// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
var bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Assign a public policy to the bucket if it exist.
if(bucketExists)
{
    // Define a public policy for the bucket.
    string publicPolicy = @"
    {
        ""Version"":""2012-10-17"",
        ""Statement"":[
            {
                ""Effect"":""Allow"",
                ""Principal"":{""AWS"":[""*""]},
                ""Action"":[
                    ""s3:GetBucketLocation"",
                    ""s3:ListBucket"",
                    ""s3:ListBucketMultipartUploads""
                ],
                ""Resource"":[""arn:aws:s3:::STORAGE_BUCKET_NAME""]
            },
            {
                ""Effect"":""Allow"",
                ""Principal"":{""AWS"":[""*""]},
                ""Action"":[
                    ""s3:AbortMultipartUpload"",
                    ""s3:DeleteObject"",
                    ""s3:GetObject"",
                    ""s3:ListMultipartUploadParts"",
                    ""s3:PutObject""
                ],
                ""Resource"":[""arn:aws:s3:::STORAGE_BUCKET_NAME/*""]
            }
        ]
    }
    ".Replace("STORAGE_BUCKET_NAME", STORAGE_BUCKET_NAME);

    // Apply the public policy to the Minio bucket to allow public access.
    SetPolicyArgs setPolicyArgs = new SetPolicyArgs().WithPolicy(publicPolicy).WithBucket(STORAGE_BUCKET_NAME);
    await minioClient.SetPolicyAsync(setPolicyArgs);

    // Get the Minio bucket's policy and print it out.
    GetPolicyArgs getPolicyArgs = new GetPolicyArgs().WithBucket(STORAGE_BUCKET_NAME);
    var policy = await minioClient.GetPolicyAsync(getPolicyArgs);
    Console.WriteLine($"policy: {policy}");
}

policy: {"Version":"2012-10-17","Statement":[{"Effect":"Allow","Principal":{"AWS":["*"]},"Action":["s3:GetBucketLocation","s3:ListBucket","s3:ListBucketMultipartUploads"],"Resource":["arn:aws:s3:::videos2"]},{"Effect":"Allow","Principal":{"AWS":["*"]},"Action":["s3:GetObject","s3:ListMultipartUploadParts","s3:PutObject","s3:AbortMultipartUpload","s3:DeleteObject"],"Resource":["arn:aws:s3:::videos2/*"]}]}


---

## Upload an Object to a Bucket from a File Path

In [10]:
// Specify the bucket name and the file you want to upload
string bucketName = STORAGE_BUCKET_NAME; // Your MinIO bucket name
string filePath = @"fixtures/SampleVideo_1280x720_1mb.mp4";  // Path to the file you want to upload
string objectName = "SampleVideo_1.mp4"; // Name to be assigned to the object in the MinIO bucket

// Check if the file exists
bool fileExists = File.Exists(filePath);

// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
var bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Upload the file to the MinIO bucket if the bucket exist, and the file exists
if(bucketExists && fileExists)
{
    // Check if the object exists in the Minio bucket.
    bool objectExist = true;
    StatObjectArgs statObjectArgs = new StatObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName);
    try { ObjectStat objectStat = await minioClient.StatObjectAsync(statObjectArgs); }
    catch(ObjectNotFoundException) { objectExist = false; }
    // if(objectExist) { return; } // uncomment this to prevent updating an existing object in the bucket

    PutObjectArgs putObjectArgs = new PutObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithFileName(filePath).WithObject(objectName);
    //.WithContentType("application/octet-stream");
    //.WithContentType("video/mp4");
    await minioClient.PutObjectAsync(putObjectArgs);
}

---

## Upload an Object to a Bucket from a File Stream

In [11]:
// Specify the bucket name and the file you want to upload
string bucketName = STORAGE_BUCKET_NAME; // Your MinIO bucket name
string filePath = @"fixtures/SampleVideo_1280x720_1mb.mp4";  // Path to the file you want to upload
string objectName = "SampleVideo_2.mp4"; // Name to be assigned to the object in the MinIO bucket

// Check if the file exists
bool fileExists = File.Exists(filePath);

// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
var bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Upload the file to the MinIO bucket if the bucket exist, and the file exists
if(bucketExists && fileExists)
{
    // Check if the object exists in the Minio bucket.
    bool objectExist = true;
    StatObjectArgs statObjectArgs = new StatObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName);
    try { ObjectStat objectStat = await minioClient.StatObjectAsync(statObjectArgs); }
    catch(ObjectNotFoundException) { objectExist = false; }
    // if(objectExist) { return; } // uncomment this to prevent updating an existing object in the bucket
    
    using (FileStream fileStream = File.OpenRead(filePath))
    {
        // Get file size
        long fileSize = fileStream.Length;
        PutObjectArgs putObjectArgs = new PutObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithStreamData(fileStream).WithObject(objectName).WithObjectSize(fileSize);
        //.WithContentType("application/octet-stream");
        //.WithContentType("video/mp4");
        await minioClient.PutObjectAsync(putObjectArgs);
    }
}

---

## List Buckets and Objects

In [12]:
// List Buckets
Console.WriteLine("Buckets\n-------");
var result = await minioClient.ListBucketsAsync();
foreach (var bucket in result.Buckets)
{
    Console.WriteLine(bucket.Name);
}

Console.WriteLine();

// List Object in one Bucket
Console.WriteLine($"Objects in bucket '{STORAGE_BUCKET_NAME}'\n---------------------------------");
ListObjectsArgs listObjectsArgs = new ListObjectsArgs().WithBucket(STORAGE_BUCKET_NAME).WithRecursive(true);
await foreach (var obj in minioClient.ListObjectsEnumAsync(listObjectsArgs))
{
    Console.WriteLine($"{obj.Key} (size: {obj.Size})");
}

Buckets
-------
videos
videos2

Objects in bucket 'videos2'
---------------------------------
SampleVideo_1.mp4 (size: 1055736)
SampleVideo_2.mp4 (size: 1055736)


---

## List Object Details

In [13]:
// Create a StatObjectArgs instance with the bucket name and the object name
string objectName = "SampleVideo_1.mp4"; // Name of object in MinIO bucket
StatObjectArgs statObjectArgs = new StatObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName);

try
{
    // Get the object details from the bucket (StatObjectAsync() throws an ObjectNotFoundException is the object doesn't exist)
    ObjectStat objectStat = await minioClient.StatObjectAsync(statObjectArgs);
    Console.WriteLine($"The object '{objectName}' in bucket '{STORAGE_BUCKET_NAME}' has the following properties ...");
    Console.WriteLine($"ObjectStat.Size: {objectStat.Size}");
    Console.WriteLine($"ObjectStat.ContentType: {objectStat.ContentType}");
    Console.WriteLine($"ObjectStat: {objectStat}");
}
catch(ObjectNotFoundException)
{
    Console.WriteLine($"The object '{objectName}' was not found in the bucket '{STORAGE_BUCKET_NAME}'");
}

The object 'SampleVideo_1.mp4' in bucket 'videos2' has the following properties ...
ObjectStat.Size: 1055736
ObjectStat.ContentType: application/octet-stream
ObjectStat: SampleVideo_1.mp4 : VersionId(None) Size(1055736) LastModified(2/13/2025 1:57:31 PM) ETag(d55bddf8d62910879ed9f605522149a8) Content-Type(application/octet-stream)
Expiry(None) ObjectLock(None) LegalHold(None) Tagging-Count(0) Archive Status(None) Replication Status(None)


---

## Download an Object from a Bucket to a File Path

In [14]:
// Specify the bucket name and the file you want to upload
string bucketName = STORAGE_BUCKET_NAME; // Your MinIO bucket name
string filePath = @"fixtures/SampleVideo_1.mp4";  // Path to the file you want to upload
string objectName = "SampleVideo_1.mp4"; // Name to be assigned to the object in the MinIO bucket

// Ensure directory exists.
string? directory = Path.GetDirectoryName(filePath);
if (!string.IsNullOrEmpty(directory) && !Directory.Exists(directory))
{
    Directory.CreateDirectory(directory);
}

// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
var bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Download the file from the MinIO bucket if the bucket exist
if(bucketExists)
{
    // Check if the object exists in the Minio bucket.
    bool objectExist = true;
    StatObjectArgs statObjectArgs = new StatObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName);
    try { ObjectStat objectStat = await minioClient.StatObjectAsync(statObjectArgs); }
    catch(ObjectNotFoundException) { objectExist = false; }

    // Download the file from the MinIO bucket if the file exist
    if(objectExist)
    {   
        GetObjectArgs getObjectArgs = new GetObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName).WithFile(filePath);
        await minioClient.GetObjectAsync(getObjectArgs);
    }
}

---

## Download an Object from a Bucket to a File Stream

In [17]:
// Specify the bucket name and the file you want to upload
string bucketName = STORAGE_BUCKET_NAME; // Your MinIO bucket name
string filePath = @"fixtures/SampleVideo_2.mp4";  // Path to the file you want to upload
string objectName = "SampleVideo_2.mp4"; // Name to be assigned to the object in the MinIO bucket

// Ensure directory exists.
string? directory = Path.GetDirectoryName(filePath);
if (!string.IsNullOrEmpty(directory) && !Directory.Exists(directory))
{
    Directory.CreateDirectory(directory);
}

// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
var bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Download the file from the MinIO bucket if the bucket exist
if(bucketExists)
{
    // Check if the object exists in the Minio bucket.
    bool objectExist = true;
    StatObjectArgs statObjectArgs = new StatObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName);
    ObjectStat objectStat = null;
    try { objectStat = await minioClient.StatObjectAsync(statObjectArgs); }
    catch(ObjectNotFoundException) { objectExist = false; }

    // Download the file from the MinIO bucket if the file exist
    if(objectExist)
    {
        GetObjectArgs getObjectArgs = new GetObjectArgs()
                                            .WithBucket(STORAGE_BUCKET_NAME)
                                            .WithObject(objectName)
                                            .WithOffsetAndLength(0L, objectStat!.Size)
                                            .WithCallbackStream(stream =>
                                            {
                                                using (var fileStream = new FileStream(filePath, FileMode.Create))
                                                {
                                                    stream.CopyTo(fileStream);
                                                }
                                            });

        await minioClient.GetObjectAsync(getObjectArgs);
    }
}

---

## Delete an Object from a Bucket

In [18]:
string bucketName = STORAGE_BUCKET_NAME; // MinIO bucket name
string[] objectNames = [ // Names of objects to be deleted in the bucket
    "SampleVideo_1.mp4",
    "SampleVideo_2.mp4"
];

// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
var bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Delete the objects from the MinIO bucket if the bucket exist
if(bucketExists)
{
    foreach (string objectName in objectNames)
    {
        // Check if the object exists in the Minio bucket.
        bool objectExist = true;
        StatObjectArgs statObjectArgs = new StatObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName);
        try { ObjectStat objectStat = await minioClient.StatObjectAsync(statObjectArgs); }
        catch(ObjectNotFoundException) { objectExist = false; }

        // Delete the object from the MinIO bucket if the object exist
        if(objectExist)
        {
            RemoveObjectArgs removeObjectArgs = new RemoveObjectArgs().WithBucket(STORAGE_BUCKET_NAME).WithObject(objectName);
            await minioClient.RemoveObjectAsync(removeObjectArgs);
        }
    }
}

---

## Delete a Bucket

In [19]:
// Check if the Minio bucket exists.
BucketExistsArgs bucketExistsArgs = new BucketExistsArgs().WithBucket(STORAGE_BUCKET_NAME);
var bucketExists = await minioClient.BucketExistsAsync(bucketExistsArgs);

// Delete the Minio bucket if it exist
if(bucketExists)
{
    // Delete the Minio bucket.
    RemoveBucketArgs removeBucketArgs = new RemoveBucketArgs().WithBucket(STORAGE_BUCKET_NAME);
    await minioClient.RemoveBucketAsync(removeBucketArgs);
}

---

## Delete Downloaded Files

In [20]:
string[] filePaths = [ // Path to files to be deleted from the file system
    @"fixtures/SampleVideo_1.mp4",
    @"fixtures/SampleVideo_2.mp4"
];

// Delete the files from the file system
foreach(string filePath in filePaths)
{
    if (File.Exists(filePath))
    {
        File.Delete(filePath);
    }
}

---

## Stop and Remove the Minio Container

In [21]:
!docker stop minio
# !docker rm minio
!docker network rm minio
!docker network ls
!docker ps

minio
minio
NETWORK ID     NAME             DRIVER    SCOPE
CONTAINER ID   IMAGE          COMMAND                  CREATED       STATUS       PORTS     NAMES
